# Chapter 2 — Build It Yourself: Micrograd

This is the hands-on heart of [Chapter 2](../README.md). You'll do the two things that
make backpropagation *click* — measure a derivative by nudging, and **backprop a graph
by hand** — then build the engine and train a neural net with it.

**Cell types:** **✍️ Your turn** (write the missing line; hint in the cell, full answer
one click away under "Stuck? Show the answer") · **📖 Study & run** (the hard cells,
shown in full) · **▶️ Run / Check your work**. Work top to bottom. Let's go. 🚀

## Step 1 — A derivative is just "nudge and measure"  ✍️ Your turn

A derivative answers: *if I nudge the input a tiny bit, how fast does the output change?*
So: take the **change in `f`** when you nudge `x` by a tiny `h`, and divide by `h`.

**Your task:** compute `slope` for `f(x) = x²` at `x = 3`. It should come out ≈ 6.

<details><summary>Stuck? Show the answer</summary>

```python
slope = (f(x + h) - f(x)) / h
```
</details>

In [ ]:
def f(x):
    return x ** 2

x, h = 3.0, 0.001
# ✍️ estimate the derivative: divide the *change in f* by the *size of the nudge* h
slope = 0.0
print(f"derivative of x^2 at x={x} ≈ {slope}   (exact answer is 2x = 6)")

In [ ]:
# ▶️ Check your work
try:
    assert abs(slope - 6.0) < 0.01, f"expected ≈ 6, got {slope}"
    print(f"✅ slope ≈ {slope:.3f} ≈ 6. A derivative really is just 'nudge and measure'.")
except (NameError, AssertionError) as e:
    print("❌", e)

## Step 2 — Backprop by hand 🔑 (the most important step)  ✍️ Your turn

The graph: `e = a*b`, then `d = e + c`, with `a=2, b=-3, c=10` (so `e=-6, d=4`). Fill in
each gradient, working **right to left**:
- through `+` (`d = e + c`): addition passes the gradient straight through.
- through `*` (`e = a*b`): each input's grad = the **other** input's value × `e_grad`.

<details><summary>Stuck? Show the answer</summary>

```python
e_grad = d_grad        # 1.0  (+ passes through)
c_grad = d_grad        # 1.0
a_grad = b_data * e_grad   # -3 * 1 = -3
b_grad = a_data * e_grad   #  2 * 1 =  2
```
</details>

In [ ]:
a_data, b_data, c_data = 2.0, -3.0, 10.0
e_data = a_data * b_data        # -6
d_data = e_data + c_data        #  4

d_grad = 1.0                    # d's gradient w.r.t. itself (given)
# ✍️ through the '+' node (d = e + c): addition passes the gradient straight through
e_grad = 0.0
c_grad = 0.0
# ✍️ through the '*' node (e = a*b): each input gets the OTHER input's value, times e_grad
a_grad = 0.0
b_grad = 0.0
print(f"a_grad={a_grad}  b_grad={b_grad}  c_grad={c_grad}")

In [ ]:
# ▶️ Check your work
try:
    assert e_grad == 1.0 and c_grad == 1.0, f"+ passes through: e_grad, c_grad should be 1, 1 (got {e_grad}, {c_grad})"
    assert abs(a_grad + 3.0) < 1e-9, f"a_grad should be b*e_grad = -3 (got {a_grad})"
    assert abs(b_grad - 2.0) < 1e-9, f"b_grad should be a*e_grad = 2 (got {b_grad})"
    print("✅ You backpropagated by hand! a_grad=-3, b_grad=2, c_grad=1 — that IS backprop.")
except (NameError, AssertionError) as e:
    print("❌", e)

## Step 3 — Build the engine: the two `_backward` rules  ✍️ Your turn

The `Value` class below is complete except for the **two most important lines in the
chapter** — the `_backward` rules for `+` and `*`. They use the same local rules you
applied by hand in Step 2:
- **`+`** passes the gradient straight through (local derivative 1).
- **`*`** gives each input the *other* input's value, times `out.grad`.

⚠️ **One change from Step 2: use `+=`, not `=`.** A `Value` can feed several places at
once, so its gradient is the *sum* of every path's contribution — we **accumulate**.
(More in lesson §6; you'll prove it in exercise E03.) The `__pow__` and `tanh` methods
just below already show the `self.grad += <local derivative> * out.grad` shape — copy it.

<details><summary>Stuck? Show the answer</summary>

For `__add__._backward`:
```python
self.grad += 1.0 * out.grad
other.grad += 1.0 * out.grad
```
For `__mul__._backward`:
```python
self.grad += other.data * out.grad
other.grad += self.data * out.grad
```
</details>

In [ ]:
import math

class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            # ✍️ '+' passes the gradient straight through:
            #    add out.grad to BOTH self.grad and other.grad
            pass
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            # ✍️ for a*b, each input's grad gets the OTHER input's value (its .data),
            #    times out.grad
            pass
        out._backward = _backward
        return out

    def __pow__(self, other):                       # given for you
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f"**{other}")
        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):                                 # given for you
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")
        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def backward(self):                             # given for you (topological sort + reverse walk)
        topo, visited = [], set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other ** -1
    def __repr__(self): return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

print("Value class defined.")

In [ ]:
# ▶️ Check your work — two graphs. The second reuses a variable, so it only comes out
#    right if your _backward ACCUMULATES with += (not =).
a, b, c = Value(2.0), Value(-3.0), Value(10.0)
d = a * b + c
d.backward()
a2 = Value(3.0)
e2 = a2 * a2          # a2 feeds the multiply twice → its grad must add up to 2a = 6
e2.backward()
try:
    assert abs(a.grad + 3.0) < 1e-9 and abs(b.grad - 2.0) < 1e-9 and abs(c.grad - 1.0) < 1e-9, \
        f"graph 1: got a.grad={a.grad}, b.grad={b.grad}, c.grad={c.grad}; expected -3, 2, 1"
    assert abs(a2.grad - 6.0) < 1e-9, \
        f"a*a should give a.grad = 2a = 6, but got {a2.grad} — did you use += (accumulate) and not = ?"
    print("✅ Your engine backprops correctly AND accumulates (a*a → 6). The engine works!")
except (NameError, AssertionError) as e:
    print("❌ Not quite:", e, "\n   (+ passes the gradient through; * swaps the inputs' values; and use += not =)")

## Step 4 — Build a neuron  ✍️ Your turn

A neuron computes `tanh(w · x + b)`: a weighted sum of its inputs plus a bias, then a
squash. Fill in the forward pass.

<details><summary>Stuck? Show the answer</summary>

The clear way — a loop that adds each weight × input to the bias, then squashes:
```python
act = self.b
for wi, xi in zip(self.w, x):
    act = act + wi * xi
return act.tanh()
```
The slick one-liner that does the exact same thing (optional):
```python
act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
return act.tanh()
```
</details>

In [ ]:
import random
random.seed(42)

class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))
    def __call__(self, x):
        # ✍️ compute the activation w·x + b, then squash it with .tanh()
        #    (a loop over zip(self.w, x), adding each wi*xi to the bias, is the clear way)
        act = self.b        # start from the bias, then add each weight × input
        return act          # then squash with .tanh()
    def parameters(self):
        return self.w + [self.b]

print(Neuron(2)([1.0, -2.0]))

In [ ]:
# ▶️ Check your work
try:
    o = Neuron(2)([1.0, -2.0])
    assert isinstance(o, Value), "the neuron should return a Value (did you call .tanh()?)"
    assert -1.0 < o.data < 1.0, "a tanh output must be in (-1, 1)"
    assert len(Neuron(2).parameters()) == 3, "a 2-input neuron has 3 params (2 weights + 1 bias)"
    print(f"✅ Neuron works — output {o.data:.3f} is a Value in (-1, 1).")
except (NameError, AssertionError, TypeError) as e:
    print("❌ Not quite:", e)

## Step 5 — Stack into an MLP and train it  📖 Study & run

Layers stack neurons; an MLP stacks layers. The training loop is the **exact same one as
Chapter 1** — forward, backward, update — but now `loss.backward()` is *your* engine.
Run it and watch the loss fall toward zero.

In [ ]:
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sizes = [nin] + nouts
        self.layers = [Layer(sizes[i], sizes[i + 1]) for i in range(len(nouts))]
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0

random.seed(1337)
xs = [[2.0,3.0,-1.0], [3.0,-1.0,0.5], [0.5,1.0,1.0], [1.0,1.0,-1.0]]
ys = [1.0, -1.0, -1.0, 1.0]
model = MLP(3, [4, 4, 1])

for step_i in range(100):
    ypred = [model(x) for x in xs]
    loss = sum((yp - yt) ** 2 for yt, yp in zip(ys, ypred))
    model.zero_grad()
    loss.backward()
    for p in model.parameters():
        p.data += -0.05 * p.grad
    if step_i % 20 == 0 or step_i == 99:
        print(f"step {step_i:3d} | loss {loss.data:.4f}")

print("targets:    ", ys)
print("predictions:", [round(model(x).data, 3) for x in xs])

In [ ]:
# ▶️ Check your work
try:
    assert loss.data < 0.1, f"after training the loss should be well under 0.1, got {loss.data:.4f}"
    print(f"✅ It learned! Final loss {loss.data:.4f}. Your hand-built autograd engine just trained a neural network. 🎉")
except (NameError, AssertionError) as e:
    print("❌", e)

## 🎓 You built backpropagation.

`loss.backward()` is no longer magic — it's the topological walk + chain rule you wrote.
The same engine, scaled up to tensors, trains every model in the rest of this course.

**Next:** the [exercises](../exercises/) (extend the engine, gradient-check vs PyTorch),
the [mini-project](../project/) (*Curve Forge*), then
[Chapter 3 — the N-gram MLP](../../03-ngram-mlp/), where we move from scalars to tensors.
See you there. 👋